# Time series examples

This notebook generates illustrative time-series paths for the white noise, moving average, and random walk examples in `src/essays/time-series.md` and saves a combined high-resolution PNG figure into `public/images/time-series-examples.png`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["figure.figsize"] = (10, 8)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

T = 500  # number of time points
SIGMA2 = 1.0  # noise variance

PROJECT_ROOT = Path("..").resolve().parent
IMAGES_DIR = PROJECT_ROOT / "public" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = IMAGES_DIR / "time-series-1-examples.png"

In [17]:
def generate_uniform_white_noise(T: int, low: float, high: float, rng: np.random.Generator) -> pd.Series:
    """Generate white noise U_t from a uniform distribution on [low, high].

    Returns a pandas Series indexed by t = 1, ..., T.
    """
    values = rng.uniform(low=low, high=high, size=T)
    time_index = np.arange(1, T + 1)
    return pd.Series(values, index=time_index, name="uniform_white_noise")


def generate_white_noise(T: int, sigma2: float, rng: np.random.Generator) -> pd.Series:
    """Generate Gaussian white noise W_t with mean 0 and variance sigma2.

    Returns a pandas Series indexed by t = 1, ..., T.
    """
    std = np.sqrt(sigma2)
    values = rng.normal(loc=0.0, scale=std, size=T)
    time_index = np.arange(1, T + 1)
    return pd.Series(values, index=time_index, name="gaussian_white_noise")


def generate_moving_average_from_innovations(innovations: pd.Series) -> pd.Series:
    """Generate the moving average process X_t = (W_{t-1} + W_t)/2 from a given innovations series.

    The input innovations series should be indexed by t = 1, ..., T and represents W_t.
    We construct X_t for t = 1, ..., T using W_{t-1} and W_t, where W_0 is taken as 0.
    """
    w = innovations.to_numpy()
    w0 = np.array([0.0])
    w_extended = np.concatenate([w0, w])
    x = 0.5 * (w_extended[:-1] + w_extended[1:])
    time_index = innovations.index
    return pd.Series(x, index=time_index, name="moving_average")


def generate_random_walk(T: int, sigma2: float, rng: np.random.Generator) -> pd.Series:
    """Generate the random walk X_t = sum_{i=1}^t W_i with X_0 = 0.

    W_i is white noise with mean 0 and variance sigma2.
    """
    std = np.sqrt(sigma2)
    w = rng.normal(loc=0.0, scale=std, size=T)
    x = np.cumsum(w)
    time_index = np.arange(1, T + 1)
    return pd.Series(x, index=time_index, name="random_walk")


def generate_random_walk(T: int, sigma2: float, rng: np.random.Generator) -> pd.Series:
    """Generate the random walk X_t = sum_{i=1}^t W_i with X_0 = 0.

    W_i is white noise with mean 0 and variance sigma2.
    """
    std = np.sqrt(sigma2)
    w = rng.normal(loc=0.0, scale=std, size=T)
    x = np.cumsum(w)
    time_index = np.arange(1, T + 1)
    return pd.Series(x, index=time_index, name="random_walk")

In [18]:
def plot_time_series_examples(
    uniform_white_noise: pd.Series,
    gaussian_white_noise: pd.Series,
    moving_average: pd.Series,
    random_walk: pd.Series,
    out_path: Path,
) -> None:
    """Plot uniform and Gaussian white noise plus the moving average and random walk.

    Layout: 2 x 2 subplots.
    - Top row: sample paths for uniform and Gaussian white noise (Example 1, parts 1 and 2).
    - Bottom row: sample paths for the moving average and random walk processes.
    """
    fig, axes = plt.subplots(2, 2, sharex="col")

    # Slightly darker soft blues
    blue_light = "#9ecae1"
    blue_mid = "#6baed6"
    blue_line = "#3182bd"
    blue_dark = "#08519c"

    # Top-left: uniform white noise (labelled as W_t(0, sigma^2))
    ax = axes[0, 0]
    ax.plot(uniform_white_noise.index, uniform_white_noise.values, color=blue_light)
    ax.set_title(r"$W_t(0, \sigma^2)$")
    ax.set_ylabel(r"$W_t$")

    # Top-right: Gaussian white noise
    ax = axes[0, 1]
    ax.plot(gaussian_white_noise.index, gaussian_white_noise.values, color=blue_mid)
    ax.set_title(r"$W_t \sim \mathcal{N}(0, \sigma^2)$")
    ax.set_ylabel(r"$W_t$")

    # Bottom-left: moving average path
    ax = axes[1, 0]
    ax.plot(moving_average.index, moving_average.values, color=blue_line)
    ax.set_title(r"$X_t = (W_{t-1} + W_t)/2$")
    ax.set_xlabel("t")
    ax.set_ylabel(r"$X_t$")

    # Bottom-right: random walk path
    ax = axes[1, 1]
    ax.plot(random_walk.index, random_walk.values, color=blue_dark)
    ax.set_title(r"$X_t = X_{t-1} + W_t$")
    ax.set_xlabel("t")
    ax.set_ylabel(r"$X_t$")

    fig.tight_layout()

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [19]:
# Uniform white noise with mean 0 and variance 1: Unif(-sqrt(3), sqrt(3))
low, high = -np.sqrt(3 * SIGMA2), np.sqrt(3 * SIGMA2)
uniform_white_noise = generate_uniform_white_noise(T=T, low=low, high=high, rng=rng)

gaussian_white_noise = generate_white_noise(T=T, sigma2=SIGMA2, rng=rng)

# Moving average built from the uniform innovations (Example 1, part 1 series)
moving_average = generate_moving_average_from_innovations(uniform_white_noise)

# Random walk driven by Gaussian white noise
random_walk = generate_random_walk(T=T, sigma2=SIGMA2, rng=rng)

plot_time_series_examples(
    uniform_white_noise=uniform_white_noise,
    gaussian_white_noise=gaussian_white_noise,
    moving_average=moving_average,
    random_walk=random_walk,
    out_path=OUTPUT_PATH,
)

OUTPUT_PATH

PosixPath('/Users/siddharthkulkarni/Desktop/GITHUB/siddharthskulkarni.github.io/public/images/time-series-examples.png')

In [20]:
summary = pd.DataFrame(
    {
        "mean": [white_noise.mean(), moving_average.mean(), random_walk.mean()],
        "std": [white_noise.std(ddof=1), moving_average.std(ddof=1), random_walk.std(ddof=1)],
    },
    index=["white_noise", "moving_average", "random_walk"],
)
summary

,mean,std
white_noise,-0.013126,0.959934
moving_average,-0.015004,0.700904
random_walk,4.559167,5.075660
